In [ ]:
# ===================== Imports =====================
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd

from collections import defaultdict
from scipy.stats import norm, pearsonr

from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from pathlib import Path
from matplotlib.gridspec import GridSpec

from scipy.spatial.distance import pdist

from tqdm import tqdm_notebook
from tqdm.notebook import trange, tqdm

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.runners_v2 import (
    run_experiment_grid,
    run_experiment_scores,
    run_experiment_scores_itemwise,
    run_experiment_itemwise_hits_fas,
    make_noise_schedule
)

from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *
from encoders import *


def load_config(cfg_path=None):
    """
    Load YAML config.
    Priority:
      1. cfg_path argument (if provided)
      2. sys.argv[1]
    """
    if cfg_path is None:
        if len(sys.argv) != 2:
            raise RuntimeError("Usage: python main.py path/to/run.yaml")
        cfg_path = sys.argv[1]

    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    with open(cfg_path, "r") as f:
        return yaml.safe_load(f), cfg_path

import itertools
import numpy as np

def log_grid(lo, hi, n):
    """Deterministic log-spaced grid between lo and hi (inclusive-ish)."""
    lo = max(float(lo), 1e-12)
    hi = max(float(hi), lo * 1.0001)
    if n <= 1 or lo == hi:
        return [lo]
    return list(np.exp(np.linspace(np.log(lo), np.log(hi), n)))

def is_valid(params, noise_mode):
    """Regime constraints (edit if your ordering assumption differs)."""
    if noise_mode == "two-regime":
        # enforce sigma0 > sigma1
        return params["sigma0"] > params["sigma1"]
    if noise_mode == "three-regime":
        # enforce sigma0 > sigma1 > sigma2
        return params["sigma0"] > params["sigma1"] > params["sigma2"]
    return True

def make_grid(lo, hi, n, *, spacing="log"):
    """
    Create 1D grid between lo and hi.

    spacing:
        - "log"    : log-spaced
        - "linear" : linearly spaced
        - "hybrid" : half log, half linear
    """
    lo = float(lo)
    hi = float(hi)

    if n <= 1 or lo == hi:
        return [lo]

    if spacing == "log":
        lo = max(lo, 1e-12)
        hi = max(hi, lo * 1.0001)
        return list(np.exp(np.linspace(np.log(lo), np.log(hi), n)))

    if spacing == "linear":
        return list(np.linspace(lo, hi, n))

    if spacing == "hybrid":
        n1 = n // 2
        n2 = n - n1
        log_part = np.exp(
            np.linspace(np.log(max(lo, 1e-12)), np.log(hi), n1, endpoint=False)
        )
        lin_part = np.linspace(lo, hi, n2)
        return sorted(set(log_part.tolist() + lin_part.tolist()))

    raise ValueError(f"Unknown spacing: {spacing}")

def generate_candidates(
    param_bounds,
    *,
    free_keys,
    fixed_params=None,
    n_per_dim=5,
    noise_mode="three-regime",
    spacing="log",          # NEW
    spacing_by_key=None,    # OPTIONAL PER-PARAM CONTROL
):
    """
    Build deterministic param dicts.

    spacing:
        "log", "linear", or "hybrid"

    spacing_by_key:
        dict like {"sigma0": "linear", "sigma1": "log"}
    """
    fixed_params = {} if fixed_params is None else dict(fixed_params)
    spacing_by_key = {} if spacing_by_key is None else dict(spacing_by_key)

    values_by_key = {}
    for k, (lo, hi) in param_bounds.items():
        if k in fixed_params:
            values_by_key[k] = [float(fixed_params[k])]
        elif k in free_keys:
            sp = spacing_by_key.get(k, spacing)
            values_by_key[k] = make_grid(lo, hi, n_per_dim, spacing=sp)
        else:
            if lo != hi:
                raise ValueError(
                    f"Key '{k}' not in free_keys and not fixed_params, but bounds vary: {(lo, hi)}"
                )
            values_by_key[k] = [float(lo)]

    keys = list(values_by_key.keys())
    grids = [values_by_key[k] for k in keys]

    candidates = []
    for vals in itertools.product(*grids):
        p = {k: v for k, v in zip(keys, vals)}
        if is_valid(p, noise_mode):
            candidates.append(p)

    return candidates

def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    """
    Estimate the median pairwise distance under a given metric.

    Args:
        X (np.ndarray): shape (N, D). Must already be preprocessed
                        appropriately for the metric
                        (e.g., L2-normalized for cosine).
        metric (str): distance metric for scipy.spatial.distance.pdist
                      ("cosine", "euclidean", "mahalanobis", etc.).
        n_samples (int): number of items to subsample for efficiency.
        seed (int): RNG seed.

    Returns:
        float: median pairwise distance.
    """
    rng = np.random.default_rng(seed)
    N = X.shape[0]

    idx = rng.choice(N, size=min(n_samples, N), replace=False)
    Xs = X[idx]

    dists = pdist(Xs, metric=metric)
    d50 = np.median(dists)

    return float(d50)


tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

In [ ]:
model_cfg, model_cfg_path = load_config("/om2/user/bjmedina/auditory-memory/memory/model_yamls/three-regime/kell2018/nontime_avg/run_000004.yaml")

noise_cfg = model_cfg["noise_model"]
#print(model_cfg['experiment']['n_seqs'])
print(model_cfg, model_cfg_path)

assert "t_step" in noise_cfg, "t_step is needed for two-regime"

In [ ]:
# ---------------------------
# SETTINGS (from YAML)
# ---------------------------
# ---- experiment ----
exp_cfg = model_cfg["experiment"]

which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)

if is_multi:
    isis = [0, 1, 2, 4, 8, 16, 32, 64]
else:
    assert which_isi is not None, "which_isi required if not multi-ISI"
    isis = [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
print(noise_cfg)

# ---- representation ----
repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]


encoder_type = repr_cfg["type"]
layer   = repr_cfg.get("layer", None)
PC_DIMS = repr_cfg.get("pc_dims", None)

# ---------------------------
# 1. Load data
# ---------------------------
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(
    which_task, which_isi, is_multi, old=False)

# ---------------------------
# 2. Human curve
# ---------------------------
human_curve = compute_human_curve(human_runs, is_multi, which_isi)
results_root = model_cfg["results_root"]
tag = model_cfg.get("tag", "untagged")

if noise_mode == "two-regime" or noise_mode == "three-regime":
    assert "t_step" in noise_cfg, f"{noise_mode} requires t_step"
    t_step = noise_cfg["t_step"]
    noise_tag = f"{noise_mode}_t{t_step}"
else:
    noise_tag = noise_mode

In [ ]:
import time

if time_avg:
    save_figs = (
        f"{results_root}/"
        f"figures/test_ISI1-and-on/v13_three-regime_time_avg/"
        f"{task_name}/"
        f"{encoder_type}/"
        f"{metric}/"
        f"{noise_tag}/"
    )
    
    save_fits = f"{results_root}/fits/test_ISI1-and-on/v13_three-regime_time_avg"
    save_fits_all = f"{results_root}/test_ISI1-and-on/test/v13_three-regime_time_avg_all"
    
    ensure_dir(save_figs)
    ensure_dir(save_fits_all)
    ensure_dir(save_fits)
else:
    save_figs = (
        f"{results_root}/"
        f"figures/test_ISI1-and-on/v13_three-regime_nontime_avg/"
        f"{task_name}/"
        f"{encoder_type}/"
        f"{metric}/"
        f"{noise_tag}/"
    )
    
    save_fits = f"{results_root}/fits/test_ISI1-and-on/v13_three-regime_nontime_avg"
    save_fits_all = f"{results_root}/fits/test_ISI1-and-on/v13_three-regime_nontime_avg_all"
    
    ensure_dir(save_figs)
    ensure_dir(save_fits_all)
    ensure_dir(save_fits)    


encoder_type = repr_cfg["type"]     # e.g. "resnet50"
layer        = repr_cfg.get("layer")
pc_dims      = repr_cfg.get("pc_dims")

print(layer)

NN_ENCODERS = {"kell2018", "resnet50"}

encoder_task = (
    "word_speaker_audioset"
    if encoder_type in NN_ENCODERS
    else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,      # e.g. "resnet50"
    model_name=encoder_type,        # same by design
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

# ---- representation-specific fields ----
if encoder_type in ("kell2018", "resnet50"):
    encoder_cfg["layer"] = layer
    assert layer is not None, f"layer required for {encoder_type}"

if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims
    assert pc_dims is not None, "pc_dims required for texture encoder"

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
print(torch.isnan(X).any())
# X is torch tensor at this point
X_np = X.detach().cpu().numpy()
print(X_np.shape)

d50 = median_pairwise_distance(
    X_np,
    metric="cosine",      # e.g. "cosine", "euclidean", "mahalanobis"
    n_samples=500,
    seed=0,
)

noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"]*1, noise_cfg["sigma0_max"]*1),
}

if noise_mode == "two-regime":
    param_bounds["sigma1"] = (
        noise_cfg["sigma1_min"]*d50,
        noise_cfg["sigma1_max"]*d50,
    )
    param_bounds["t_step"] = (
        noise_cfg["t_step"],
        noise_cfg["t_step"],
    )

if noise_mode == "three-regime":
    param_bounds["sigma1"] = (
        noise_cfg["sigma1_min"]* d50,
        noise_cfg["sigma1_max"]* d50,
    )

    param_bounds["sigma2"] = (
        noise_cfg["sigma2_min"]* d50,
        noise_cfg["sigma2_max"]* d50,
    )
    param_bounds["t_step"] = (
        noise_cfg["t_step"],
        noise_cfg["t_step"],
    )


paramA = dict(param_bounds)

def log_mid(lo, hi):
    lo, hi = max(lo, 1e-12), max(hi, 1e-12)
    return float(np.exp(0.5 * (np.log(lo) + np.log(hi))))

print("median distance:", d50)

In [ ]:
def make_isi0_experiment(stim_path):
    """
    Returns a single experiment with ISI=0:
    two identical trials back-to-back.
    """
    return [stim_path, stim_path]

def make_isi0_block_experiment(stimuli):
    """
    Build an ISI=0-only experiment:
    [A, A, B, B, C, C, ...]
    """
    seq = []
    for stim in stimuli:
        seq.extend([stim, stim])
    return seq

def make_fixed_isi_experiment(stimuli, isi):
    """
    Build an experiment where every item repeats after exactly ISI intervening items.

    Example (isi=4):
    [A B C D E, A B C D E, F G H I J, F G H I J, ...]
    """
    assert isi >= 1

    block_size = isi + 1
    seq = []

    # truncate to a multiple of block_size
    n_blocks = len(stimuli) // block_size
    stimuli = stimuli[: n_blocks * block_size]

    for i in range(n_blocks):
        block = stimuli[i * block_size : (i + 1) * block_size]
        seq.extend(block)
        seq.extend(block)

    return seq

def make_fixed_isi_experiments(
    stimulus_pool,
    isi,
    K=20,
    n_exps=20,
):
    """
    Generate multiple fixed-ISI experiments.
    """
    exps = []

    block_size = isi + 1
    needed = K * block_size

    for _ in range(n_exps):
        random.shuffle(stimulus_pool)
        stimuli = stimulus_pool[:needed]

        exp = make_fixed_isi_experiment(
            stimuli=stimuli,
            isi=isi,
        )

        exps.append(exp)

    return exps

def log_mid(lo, hi):
    lo, hi = max(lo, 1e-12), max(hi, 1e-12)
    return float(np.exp(0.5 * (np.log(lo) + np.log(hi))))

def obj_ge2(r):
    v = np.asarray(r["mse_per_isi"], float)
    return float(np.nanmean(v[1:]))

def total_obj(r, w0=0.3):
    return w0 * r["mse_isi0"] + (1 - w0) * r["mse_ge1"]


paramA = dict(param_bounds)


stimulus_pool = list(set(exp_list[0]))
single_stimulus_exp_list = [
    make_isi0_experiment(stimulus_pool[0])
]

multiple_stimuli_exp_list = [
    make_isi0_experiment(stim)
    for stim in stimulus_pool
]


isi0_exp_list = []
K = 20  # number of distinct stimuli to include


for _ in range(20):
    random.shuffle(stimulus_pool)
    stimuli = stimulus_pool[:K]
    single_exp = make_isi0_block_experiment(stimuli)
    
    isi0_exp_list.append(single_exp)

ISI = 16

In [ ]:
isi4_exp_list = make_fixed_isi_experiments(
    stimulus_pool,
    isi=ISI,
    K=3,
    n_exps=5,
)

In [ ]:
len(isi4_exp_list[0]), isi4_exp_list[0]

In [ ]:
isis = [1]                 # only ISI=0
NUM_SUBSAMPLES = len(isi0_exp_list)  # no subsampling
random_draw = False        # no shuffling
trange

In [ ]:
from scipy.stats import norm

def sweep_dprime_empirical(hit_scores, fa_scores, eps=1e-6):
    """
    Compute d' values by sweeping all empirical score thresholds.
    """
    hit_scores = np.asarray(hit_scores)
    fa_scores  = np.asarray(fa_scores)

    thresholds = np.unique(np.concatenate([hit_scores, fa_scores]))

    dprimes = []

    for thr in thresholds:
        hit_rate = np.mean(hit_scores <= thr)
        fa_rate  = np.mean(fa_scores  <= thr)

        # avoid infinities
        hit_rate = np.clip(hit_rate, eps, 1 - eps)
        fa_rate  = np.clip(fa_rate,  eps, 1 - eps)

        dprimes.append(norm.ppf(hit_rate) - norm.ppf(fa_rate))

    return np.asarray(dprimes)

In [ ]:
sigma0_lo = 2.0
sigma0_hi = 3.5

N_MC = 3
N_PER_DIM = 20

paramA = dict(param_bounds)

# 🔒 restrict sigma0 to trusted regime
paramA = dict(param_bounds)
paramA["sigma0"] = (3.0, 3.0)


sigma1_curr = log_mid(*param_bounds["sigma1"])
sigma2_curr = log_mid(*param_bounds["sigma2"])

# ---- FIX sigma2 for now ----
sigma2_curr = log_mid(*param_bounds["sigma2"])
paramA["sigma2"] = (sigma2_curr, sigma2_curr)

# ---- SWEEP sigma1 ----
paramA["sigma1"] = (0.01, 3.0)

cands = generate_candidates(
    paramA,
    free_keys=["sigma1"],      # 🔑 KEY CHANGE
    n_per_dim=N_PER_DIM,
    spacing="log",
)

sigma1_vals = []

mean_hit_scores = []
std_hit_scores  = []

mean_fa_scores  = []
std_fa_scores   = []

mean_dprimes = []
std_dprimes  = []
max_dprimes  = []

for c in cands:

    sigma1 = c["sigma1"]

    hit_scores = []
    fa_scores  = []

    for rep in trange(N_MC):
        run_out = run_experiment_scores(
            sigma0=3.0,      # 🔒 fixed
            sigma1=sigma1,           # 🔁 swept
            sigma2=sigma2_curr,
            t_step=c["t_step"],
            rate=0,
            noise_mode=noise_mode,
            metric=metric,
            X0=X,
            name_to_idx=name_to_idx,
            experiment_list=isi4_exp_list,   # ISI=4 experiments
            debug=False,
            seed=rep * 2 + 1
        )

        hit_scores.extend(run_out["hits"])
        fa_scores.extend(run_out["fas"])

    sigma1_vals.append(sigma1)

    mean_hit_scores.append(np.mean(hit_scores))
    std_hit_scores.append(np.std(hit_scores))

    mean_fa_scores.append(np.mean(fa_scores))
    std_fa_scores.append(np.std(fa_scores))

    dps = sweep_dprime_empirical(hit_scores, fa_scores)

    mean_dprimes.append(np.mean(dps))
    std_dprimes.append(np.std(dps))
    max_dprimes.append(np.max(dps))

In [ ]:
plt.figure(figsize=(6, 4))
plt.errorbar(
    sigma1_vals,
    mean_dprimes,
    yerr=std_dprimes,
    fmt="o-",
    capsize=3,
    label="Mean d′ ± std (across thresholds)"
)
plt.axhline(y=human_curve[5], c='k', label="human d'", alpha=0.5)

plt.xscale("log")
plt.xlabel(r"$\sigma_1$")
plt.ylabel("d′")
plt.title(f"Threshold-swept d′ vs noise scale (ISI = {ISI})")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.errorbar(
    sigma1_vals, mean_hit_scores,
    yerr=std_hit_scores,
    label="Hits", fmt="o-", capsize=3
)
plt.errorbar(
    sigma1_vals, mean_fa_scores,
    yerr=std_fa_scores,
    label="False alarms", fmt="o-", capsize=3
)

plt.xscale("log")
plt.xlabel(r"$\sigma_1$")
plt.ylabel("Score (NND)")
plt.legend()
plt.title(f"Score means ± std vs noise scale (ISI={ISI})")
plt.tight_layout()
plt.show()

In [ ]:
human_curve[0]